In [1]:
from tabulate import tabulate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
pd.set_option('display.max_colwidth', None)


#### Question 1 : node potential for 1st test word ######
def NodePotential(dataframe_test, dataframe_test1,df):

    rows_1 = [['e'],['t'],['a'],['i'],['n'],['o'],['s'],['h'],['r'],['d']]
    headers_1 = []
    for x in range(len(dataframe_test[0][0])):
        headers_1.append(dataframe_test[0][0][x])

    for i in range(dataframe_test1.shape[0]):
      for yi in range(len(y_labels)):
          rows_1[yi].append(np.round(np.log(np.exp(np.dot(df.iloc[yi,:], dataframe_test1.iloc[i]))),3))

    print(tabulate(rows_1, headers=headers_1, tablefmt='fancy_grid'))



#### Question 2: Negative Energy #####
def NegativeEnergy(df,datafr,transition_params,word):

        term = 0
        length = len(word)
        for letter in range(length):
            term = term + np.log(np.exp(np.dot(df.iloc[y_labels.get(word[letter])['index']],datafr.iloc[letter])))
        transpot = 0
        for trans in range(length - 1):
            transpot = transpot + np.log(np.exp(transition_params.iloc[y_labels.get(word[trans])['index']][y_labels.get(word[trans+1])['index']]))
        NegEng = term+transpot

        return NegEng  
    
##### Question 3: Normalizing Constant #####
def NormalizingConstant(datafr, df,transition_params, word):
    
    length = len(word)
    norm_const = 0
    A = [0]*10
    for letter in range(length):
      if letter == 0:
        B = [1]*10
      else:
        B = []
        for b in range(10):
            B.append(np.dot(A,np.exp(transition_params.iloc[b])))    
      for a in range(10):
          A[a] = np.exp(np.dot(df.iloc[a],datafr.iloc[length-letter-1]))
      A = np.array(A)*np.array(B)
      if letter == length - 1:
        norm_const = np.sum(A)
        return norm_const
 
#### Question 4: For the first three test words, compute the most likely joint labeling (character sequence) word######
def JointProb(datafr,df,dataframe_test,transition_params,word):   
    prob = 0   
    norm_const = NormalizingConstant(datafr=datafr, df=df,transition_params=transition_params, word = word)
    values = ['e','t','a','i','n','o','s','h','r','d']
    combinations_array = [values]*len(dataframe_test[0][q4])
    combinations = np.array(np.meshgrid(*combinations_array)).T.reshape(-1, len(dataframe_test[0][q4]))
    char_seq = combinations[0]

    for comb in combinations:
        sum = 0
        for s in range(len(dataframe_test[0][q4])):
          sum = sum + np.dot(df.iloc[y_labels.get(comb[s])['index']],datafr.iloc[s])
        for t in range(len(dataframe_test[0][q4]) - 1):
          sum = sum + transition_params.iloc[y_labels.get(comb[t])['index'],:][y_labels.get(comb[t+1])['index']]
        prob_calc = (np.exp(sum))/norm_const
        if prob_calc > prob:
          prob = prob_calc 
          char_seq = comb

    return prob, char_seq

###### Question 5: For the first test word only, compute the marginal probability distribution over character labels
#### for each position in the word


def MarginalProb(dataframe_test,df,transition_params,dataframe_test1):
    
    rows_2 = [['e', 1, 1, 1, 1],['t', 1, 1, 1, 1],['a', 1, 1, 1, 1],['i', 1, 1, 1, 1],['n', 1, 1, 1, 1],['o', 1, 1, 1, 1],['s', 1, 1, 1, 1],['h', 1, 1, 1, 1],['r', 1, 1, 1, 1],['d', 1, 1, 1, 1]]   
    length_new = len(dataframe_test[0][0])
    
    ### pos represents the position for which we are calculating the marginals
    #pos = 0  
    ### i represents the summations over y0,y1,y2,y3. When i == pos, we dont sum over that position and instead find values for yi for the 10 cases
    
    for pos in range(4):
        flag = 0
        C = [0]*10
        interfactors = [[]*10]*10
        for i in range(length_new-1,-1,-1):     
            if i == pos:
                if i == length_new-1:
                  for j in range(10): 
                    rows_2[j][pos+1] = np.exp(np.dot(df.iloc[j],dataframe_test1.iloc[pos]))
                else:
                  for j in range(10):
                      xx = np.exp(np.dot(df.iloc[j],dataframe_test1.iloc[pos]))
                      yy = np.dot(C,np.exp(transition_params.iloc[j]))
                      rows_2[j][pos+1] = xx*yy
                if pos == 0:
                  continue
                else:
                  flag = 1
                  #### store const*exp(theta(x,y))
                  for constant in range(10):
                      #### store 100 intermediate factors as eg: x0*exp(theta00) xo*exp(theta10) and so on
                      interfactors[constant] = np.dot(rows_2[constant][pos+1],np.exp(transition_params.iloc[:,constant]))  
                  continue
            else:
                if i==3:
                  D = [1]*10
                elif i != pos-1: 
                    if flag == 0:                  
                      D = []
                      for b in range(10):
                          D.append(np.dot(C,np.exp(transition_params.iloc[b])))    
                    if flag == 1:
                      D = []
                      for m in range(10):
                        D = interfactors[m].copy()
                        for n in range(10):
                          interfactors[m][n] = np.dot(D,np.exp(transition_params.iloc[n])) 

                for a in range(10):
                    C[a] = np.exp(np.dot(df.iloc[a],dataframe_test1.iloc[i]))

                if flag == 1:
                    #factors = []                       
                    for xx in range(10):
                        interfactors[xx] = np.multiply(C,interfactors[xx])

                if flag == 0:
                    C = np.array(C)*np.array(D)


                #### Finally store values in rows_2 #######
                if i == 0:
                   for z in range(10):                
                       rows_2[z][pos+1] = np.sum(interfactors[z])
    return rows_2


###### question 6: Message passing  ########



def ComputeMessages(df,transition_params,datafr, word, tableq6):

    length = len(word)
    labels = 'etainoshrd'
    zeros = [0] * (length-1)

    # rows_3 = [['e', 1, 0, 0, 0],['t',1 , 0, 0, 0],['a',1 , 0, 0, 0],['i',1 , 0, 0, 0],['n',1 , 0, 0, 0],['o',1 , 0, 0, 0],['s',1 , 0, 0, 0],['h',1 , 0, 0, 0],['r',1 , 0, 0, 0],['d',1 , 0, 0, 0]]
    rows_3 = [[labels[i], 1] + zeros for i in range(len(labels))]
    #headers_3 = ['m0-1(y1)','m1-2(y2)**','m2-3(y3)','m3-4(y4)']
    headers_3 = []
    for h3 in range(length):
        headers_3.append('m'+str(h3)+'-'+str(h3+1)+'(y'+str(h3+1)+')')  

    #rows_4 = [['e', 0, 0, 0, 1],['t', 0, 0, 0, 1],['a', 0, 0, 0, 1],['i', 0, 0, 0, 1],['n', 0, 0, 0, 1],['o', 0, 0, 0, 1],['s', 0, 0, 0, 1],['h', 0, 0, 0, 1],['r', 0, 0, 0, 1],['d', 0, 0, 0, 1]]
    # headers_4 = ['m2-1(y1)**','m3-2(y2)**','m4-3(y3)**','m5-4(y4)']
    rows_4 = [[labels[i]] + zeros + [1] for i in range(len(labels))]
    headers_4 = []
    for h4 in range(length):
        headers_4.append('m'+str(h4+2)+'-'+str(h4+1)+'(y'+str(h4+1)+')')


    if tableq6 == "True":
      headers_tableq6 = ['m1-2(y2)','m2-1(y1)','m4-3(y3)','m3-2(y2)']
      rows_tableq6 = [[labels[i], 1] + zeros for i in range(len(labels))]
    
    ### Forward message passing 
    for i in range(2,length+1):
      k = i-2 
      j = i-1
      for m in range(10):
          temp = 0
          for n in range(10):
            temp = temp + rows_3[n][j]*np.exp(np.dot(df.iloc[n],datafr.iloc[k]))*np.exp(transition_params.iloc[n][m])
          rows_3[m][i] = np.log(temp)
          if  tableq6=="True": 
            if i==2:
              rows_tableq6[m][1] = np.log(temp)

    ### backward message passing 
    for i in range(length-1,0,-1):
      k = i+2 
      j = i+1  
      for m in range(10):
          temp = 0
          for n in range(10):
            temp = temp + rows_3[n][j]*np.exp(np.dot(df.iloc[n],datafr.iloc[i]))*np.exp(transition_params.iloc[m][n])
          rows_4[m][i] = np.log(temp)
          if  tableq6=="True":
            if i == 1:
              rows_tableq6[m][2] = np.log(temp)
            elif i == 2:
              rows_tableq6[m][4] = np.log(temp)
            elif i == 3:
              rows_tableq6[m][3] = np.log(temp)

    if tableq6=="True":
      return rows_3, rows_4, rows_tableq6, headers_3, headers_4, headers_tableq6
    else:
      return rows_3,rows_4, headers_3, headers_4

##### Question 7 #######
def MarginalsMessagePassing(df,datafr, rows_forward, rows_backward,word):

    #rows_5 = [['e', 1, 1, 1, 1],['t', 1, 1, 1, 1],['a', 1, 1, 1, 1],['i', 1, 1, 1, 1],['n', 1, 1, 1, 1],['o', 1, 1, 1, 1],['s', 1, 1, 1, 1],['h', 1, 1, 1, 1],['r', 1, 1, 1, 1],['d', 1, 1, 1, 1]]

    labels = 'etainoshrd'
    length = len(word)
    ones = [1] * (length)
    rows_5 = [[labels[i]] + ones for i in range(len(labels))]

    for i in range(length):
      temp = []
      sum = 0
      for j in range(10):
          t1 = rows_forward[j][i+1]
          t2 = np.dot(df.iloc[j],datafr.iloc[i])
          t3 = rows_backward[j][i+1]
          temp.append(np.exp(t1+t2+t3))
          sum = sum + temp[j]

      temp = temp/sum
      
      for k in range(10):
        rows_5[k][i+1] = temp[k]

    return rows_5


##### Qusetion 7 - pairwise marginals #######
#### e = 0, t = 1, r = 8

def PairwiseMarginals(datafr,word,transition_params,df,dataframe_test, rows_pairwise,headers_7, j, rows_output, headers_output):

     norm_constant = 0 
     for m in range(10):
       for n in range(10):
         rows_pairwise[m][n+1] = np.exp(rows_forward[n][j+1] + np.dot(df.iloc[m],datafr.iloc[j]) + np.dot(df.iloc[n],datafr.iloc[j+1]) + transition_params.iloc[m][n] + rows_backward[n][j+2])
         norm_constant = norm_constant + rows_pairwise[m][n+1]
     
     final_rows = [[row[0]] + [f"{r/norm_constant:.4e}" for r in row[1:]] for row in rows_pairwise]    
     index = [0,1,8]
     for m in range(3):
       for n in range(3):
         rows_output[m][n+1] = final_rows[index[m]][index[n]+1]

     print(tabulate(rows_output, headers=headers_output, tablefmt='fancy_grid'))


#### Question 13: Avg log likelihood ###
def AverageLogLikelihood(df,datafr, transition_params, word):
    
   # word = 
    # print(word)
    E = NegativeEnergy(df=df,datafr=datafr,transition_params=transition_params, word = word)
    #print("neg energy = ", E)
    Z_norm = NormalizingConstant(datafr=datafr, df=df,transition_params=transition_params, word = word)
    #print("Z= ",np.log(Z_norm))
    T = E - np.log(Z_norm)
    return T


##### Code START ######
#### Store labels as a dictionary
y_labels = { 'e': {'index': 0},
             't': {'index': 1},
             'a': {'index': 2},
             'i': {'index': 3},
             'n': {'index': 4},
             'o': {'index': 5},
             's': {'index': 6},          
             'h': {'index': 7},
             'r': {'index': 8},
             'd': {'index': 9},           
          }

#### Question 1 : node potential for 1st test word ######
df = pd.read_csv('../model/feature-params.txt', delimiter=' ', header=None)
dataframe_test = pd.read_csv('../data/test-words.txt', delimiter=' ', header=None)
dataframe_test1 = pd.read_csv('../data/test-img-1.txt', delimiter=' ', header=None)
print("\n #########################################################")
print("\nQuestion 1: The Node Potential for test word hair is:")
NodePotential(dataframe_test=dataframe_test,dataframe_test1=dataframe_test1,df=df)
print("\n #########################################################")

#### Question 2: Negative Energy #####
print("\n Question 2: Negative Energy")
transition_params = pd.read_csv('../model/transition-params.txt', delimiter=' ', header=None)
for x in range(3):
  word = dataframe_test[0][x]
  datafr = pd.read_csv('../data/test-img-'+str(x+1)+'.txt', delimiter=' ', header=None)
  NegEng = NegativeEnergy(df=df,datafr=datafr,transition_params=transition_params, word = word)
  print("Negative energy for the test word ", word ," is = ",np.round(NegEng,3))

print("\n #########################################################")
##### Question 3: Normalizing Constant #####
print("\nQuestion 3: Normalizing Constant:")
for q3 in range(3):
     word =  dataframe_test[0][q3]
     datafr = pd.read_csv('../data/test-img-'+str(q3+1)+'.txt', delimiter=' ', header=None)
     Z = NormalizingConstant(datafr=datafr, df=df,transition_params=transition_params, word = word)
     print("Normalizing constant for the test word ", dataframe_test[0][q3] ," is = ", np.round(np.log(Z),3))

print("\n #########################################################")
##### Question 4: Joint prob ########
print("\nQuestion 4: Joint Probability:")
for q4 in range(3):
    word = dataframe_test[0][q4]
    datafr = pd.read_csv('../data/test-img-'+str(q4+1)+'.txt', delimiter=' ', header=None)
    prob, char_seq = JointProb(datafr=datafr,df=df,dataframe_test= dataframe_test,transition_params=transition_params,word = word)
    print("The most likely joint labelling for the word ",dataframe_test[0][q4] ," is ", char_seq," with probability = ", np.round(prob,3))

print("\n #########################################################")
##### QUestion 5:Marginal Probability ######
print("\nQuestion 5: Marginal Probability:")
word = dataframe_test[0][0]
datafr = globals()['dataframe_test1']
norm_const_test1 = NormalizingConstant(datafr=datafr, df=df,transition_params=transition_params,word = word)
#print("Normalizing constant is = ", norm_const_test1)
rows_2 = MarginalProb(dataframe_test=dataframe_test,df=df,transition_params=transition_params,dataframe_test1=dataframe_test1)
final_rows = [[row[0]] + [f"{r/norm_const_test1:.4e}" for r in row[1:]] for row in rows_2]
headers_2 = []
for x in range(len(dataframe_test[0][0])):
    headers_2.append('position '+str(x))

print(tabulate(final_rows, headers=headers_2, tablefmt='fancy_grid'))

print("\n #########################################################")
##### Question 6: compute messages #####
print("\nQuestion 6: Computing Messages")
for i in range(1):
    word = dataframe_test[0][i]
    datafr = pd.read_csv('../data/test-img-'+str(i+1)+'.txt', delimiter=' ', header=None)  
    rows_3,rows_4, rows_tableq6, headers_3, headers_4, headers_tableq6 = ComputeMessages(df=df,transition_params=transition_params,datafr=datafr, word=word, tableq6 = "True")  
    print(tabulate(rows_tableq6, headers=headers_tableq6, tablefmt='fancy_grid'))

print("\n #########################################################")
#### Question 7: single marginals via message passing ######
print("\nQuestion 7: Single Marginals")
for i in range(1):
    word = dataframe_test[0][i]
    datafr = pd.read_csv('../data/test-img-'+str(i+1)+'.txt', delimiter=' ', header=None)
    rows_forward,rows_backward, h2,h3 = ComputeMessages(df=df,transition_params=transition_params,datafr=datafr, word = word,tableq6="False")
    rows_5 = MarginalsMessagePassing(df=df,datafr=datafr, rows_forward=rows_forward, rows_backward=rows_backward,word = word)
    headers_5 = []
    for x in range(len(word)):
        headers_5.append('position '+str(x))
    print(tabulate(rows_5, headers=headers_5, tablefmt='fancy_grid'))

#### Question 7: Pairwise Marginals ##### 
print("\nQuestion 7: Pairwise Marginals:")
labels = 'etainoshrd' 
headers_pairwise = ['e','t','a','i','n','o','s','h','r','d']
ones = [1] * (10)

for i in range(1):
    word = dataframe_test[0][i]
    datafr = pd.read_csv('../data/test-img-'+str(i+1)+'.txt', delimiter=' ', header=None)
    rows_forward,rows_backward, h2,h3 = ComputeMessages(df=df,transition_params=transition_params,datafr=datafr, word = word,tableq6="False")
    length = len(word)     
    for j in range(length-1):
        rows_pairwise = [['y'+str(j+1)+'='+labels[k]] + ones for k in range(len(labels))]
        headers_7 = ['y'+str(j+2)+'='+labels[k] for k in range(len(labels))]
        rows_output = [['y'+str(j+1)+'= e', 1, 1, 1], ['y'+str(j+1)+'= t',1,1,1], ['y'+str(j+1)+'= r',1,1,1]]
        headers_output = ['y'+str(j+2)+'= e', 'y'+str(j+2)+'= t' , 'y'+str(j+2)+'= r' ]
        print("Pairwise marginal for y"+str(j+1)+"y"+str(j+2)+":")
        PairwiseMarginals(datafr=datafr,word=word,transition_params=transition_params,df=df,dataframe_test=dataframe_test, rows_pairwise=rows_pairwise,headers_7=headers_7, j=j, rows_output=rows_output, headers_output=headers_output)

print("\n #########################################################")
#### Question 8: Accuracy 
print("\nQuestion 8: Character Level Accuracy:")
labels = 'etainoshrd'
total_chars = 0
correctly_predicted_chars = 0

for r in range(len(dataframe_test[0])):

  word = dataframe_test[0][r]
  if r < 5:
    print("\nThe predicted sequence for the test word "+word+" is:")
  total_chars = total_chars + len(word)
  datafr = pd.read_csv('../data/test-img-'+str(r+1)+'.txt', delimiter=' ', header=None)
  rows_forward,rows_backward, h2,h3 = ComputeMessages(df=df,transition_params=transition_params,datafr=datafr, word = word,tableq6="False")
  rows_marginals = MarginalsMessagePassing(df=df,datafr=datafr, rows_forward=rows_forward, rows_backward=rows_backward,word = word)
  rows_marginals_array = np.array(rows_marginals)
  rows_marginals_array = rows_marginals_array
  for s in range(len(word)):
    #print(rows_marginals_array[:,s+1])
    col = rows_marginals_array[:,s+1]
    col = 1e20*col.astype(float)
    pred = labels[np.argmax(col)]
    if pred == word[s]:
       correctly_predicted_chars = correctly_predicted_chars + 1
    if r<5:
      print("'"+pred+"'", end= "")


print("\n The average character level accuracy = ", np.round(correctly_predicted_chars/total_chars,3))

print("\n #########################################################")
#### Question 13- avg log likelihood ###### 
print("\nQuestion 13: Average Log Likelihood:")
log_sum = 0
dataframe_train = pd.read_csv('../data/train-words.txt', delimiter=' ', header=None)
for tr in range(50):
  word = dataframe_train[0][tr]
  datafr = pd.read_csv('../data/train-img-'+str(tr+1)+'.txt', delimiter=' ', header=None)
  T = AverageLogLikelihood(df=df,datafr=datafr, transition_params=transition_params, word = word)
  log_sum = log_sum + T


print("The average log-likelihood of the training set ",np.round(log_sum/50,4))

print("\n #########################################################")
##### Question 16: optimization #####
print("\nQuestion 16: Optimization:")

def objective_function(x):
    return (1-x[0])**2 + 50*(x[1]-x[0]**2)**2


def gradient(x):

  gradients = np.array([0,0])
  gradients[0] = -2*(1-x[0]) - 200*(x[1]-x[0]**2)*x[0] 
  gradients[1] = 100*(x[1]-x[0]**2)
  return gradients


x0 = np.array([0,0])
result = minimize(objective_function, x0 = x0, method='BFGS', jac=gradient, options={'disp': True})
print("The maxima is at = ",np.round(result.x,3))
print("The value of the objective function at the maximum is =", np.round(result.fun,3))



 #########################################################

Question 1: The Node Potential for test word hair is:
╒════╤═════════╤═════════╤════════╤═════════╕
│    │       h │       a │      i │       r │
╞════╪═════════╪═════════╪════════╪═════════╡
│ e  │ -10.227 │ -16.412 │ -2.032 │ -19.391 │
├────┼─────────┼─────────┼────────┼─────────┤
│ t  │ -12.47  │  -9.864 │  9.427 │  12.295 │
├────┼─────────┼─────────┼────────┼─────────┤
│ a  │   7.356 │  17.699 │ -2.682 │   6.319 │
├────┼─────────┼─────────┼────────┼─────────┤
│ i  │ -18.053 │  -8.418 │ 10.549 │  -1.457 │
├────┼─────────┼─────────┼────────┼─────────┤
│ n  │  14.715 │   9.829 │ -6.472 │   2.44  │
├────┼─────────┼─────────┼────────┼─────────┤
│ o  │  -1.247 │  -3.846 │ -9.94  │   0.025 │
├────┼─────────┼─────────┼────────┼─────────┤
│ s  │  -6.053 │ -10.705 │  6.588 │ -11.689 │
├────┼─────────┼─────────┼────────┼─────────┤
│ h  │  26.629 │   6.116 │ -8.161 │   0.491 │
├────┼─────────┼─────────┼────────┼─────────┤
│ r  │  -7.